# Stage-2 rescue diagnostics v1

Run all on an A100 80 GB runtime. This is an R-only diagnostic flow. It does not continue scientific training, open P:0006, or touch P:0009; its only optimization is the explicitly isolated synthetic micro-overfit gate.

In [ ]:
from pathlib import Path
import json, os, re, subprocess

REPOSITORY_URL = 'https://github.com/GuillermoTafoya/MRIxFields.git'
TRAINING_EVIDENCE_COMMIT = '82633d66e5ea47f96b149ea22cc192fcf4526f06'
RESCUE_IMPLEMENTATION_COMMIT = 'feaa91dc0b00957a14d5104d250125e2538b4d08'
if re.fullmatch(r'[0-9a-f]{40}', RESCUE_IMPLEMENTATION_COMMIT) is None:
    raise RuntimeError('Notebook is unsealed: rescue implementation commit is not pinned.')
probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], text=True, capture_output=True, check=True)
rows = [row.strip() for row in probe.stdout.splitlines() if row.strip()]
if len(rows) != 1 or not rows[0].startswith('NVIDIA A100-SXM4-80GB,'):
    raise RuntimeError('Run all requires exactly one NVIDIA A100-SXM4-80GB.')
REPO_DIR = Path('/content/MRIxFields-stage2-rescue-' + RESCUE_IMPLEMENTATION_COMMIT[:12])
if REPO_DIR.exists():
    raise FileExistsError('Pinned rescue checkout already exists; use a fresh runtime.')
subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', 'origin', RESCUE_IMPLEMENTATION_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', RESCUE_IMPLEMENTATION_COMMIT], cwd=REPO_DIR, check=True)
def git_text(*args):
    env = os.environ.copy(); env['GIT_OPTIONAL_LOCKS'] = '0'
    result = subprocess.run(['git', *args], cwd=REPO_DIR, text=True, capture_output=True, env=env, check=True)
    return result.stdout.strip()
if git_text('rev-parse', 'HEAD') != RESCUE_IMPLEMENTATION_COMMIT:
    raise RuntimeError('Detached checkout identity mismatch.')
if git_text('status', '--porcelain=v1', '--untracked-files=all'):
    raise RuntimeError('Rescue checkout is dirty.')
if subprocess.run(['git', 'merge-base', '--is-ancestor', TRAINING_EVIDENCE_COMMIT, 'HEAD'], cwd=REPO_DIR).returncode:
    raise RuntimeError('Rescue implementation does not descend from training evidence.')
from google.colab import drive
drive.mount('/content/drive')
print(json.dumps({'hardware_preflight': 'pass', 'detached_clean_checkout': True, 'training_authorized': False, 'P0006_accessed': False, 'P0009_accessed': False}, sort_keys=True), flush=True)
operator = REPO_DIR / 'notebooks/stage2_rescue_diagnostics_operator.py'
exec(compile(operator.read_text(encoding='utf-8'), str(operator), 'exec'), globals())
